# Hafta 3 · Ders 5 — Gauss Eleme ve LU Ayrışımı

> **Makine Öğrenmesi için Lineer Cebir** · ilk ilkelerden bir kurs
>
> *türet → uygula → görselleştir → doğrula → makine öğrenmesine bağla*

Bir bilgisayar $A\mathbf{x} = \mathbf{b}$'yi aslında nasıl çözer? $A$'yı tersine çevirerek değil — bu yavaş
ve sayısal olarak tehlikelidir. **Gauss eleme** kullanır: sistem üçgensel ve geri yerine koyma ile
kolayca çözülebilir hâle gelene kadar köşegenin altındaki girdileri sistematik olarak sıfırlar.

Bu süreci yeniden kullanılabilir bir çarpanlamaya şişelemek, `scipy.linalg.solve`'un arkasındaki
beygir gücü olan **LU ayrışımı** $A = LU$'yu verir. Her ikisini de elle inşa ediyor, ardından
**pivotlamanın** neden var olduğu gerçeğiyle yüzleşiyoruz: pivotlama olmadan kayan-nokta hataları patlar.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu as scipy_lu
from utils.linalg_viz import check

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.dpi"] = 110

## 1. Geri yerine koyma: üçgensel bir sistemi çözmek

Bir sistem zaten **üst-üçgensel** ise, çözmesi kolaydır: son denklemde bir bilinmeyen vardır ve
yukarı doğru çalışarak yerine koyarız. Bu, elemenin getirisi olan adımdır, bu yüzden onu
önce inşa ediyoruz.

In [2]:
def back_substitution(U, b):
    U, b = np.asarray(U, float), np.asarray(b, float)
    n = len(b)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

U = np.array([[2.0, 1.0, -1.0],
              [0.0, 3.0,  2.0],
              [0.0, 0.0,  4.0]])
b = np.array([5.0, 4.0, 8.0])
x = back_substitution(U, b)
check("geri yerine koyma", U @ x, b)
print("çözüm x =", x)

[PASS] back-substitution                max|Δ| = 0.00e+00
solution x = [3.5 0.  2. ]


## 2. Gauss eleme → LU

Eleme, pivotun altında sıfırlar oluşturmak için bir satırın bir katını altındakilerden tekrar tekrar
çıkarır. Önemli kavrayış: **kullandığımız çarpanlar tam olarak alt-üçgensel bir matris $L$'nin
girdileridir** ve geriye kalan üst-üçgensel $U$'dur. Yani eleme, çarpanlamanın *kendisidir*

$$ A = LU, \qquad L = \text{birim alt-üçgensel (çarpanlar)}, \quad U = \text{üst-üçgensel}. $$

$L$ ve $U$'ya sahip olduğumuzda, $A\mathbf{x}=\mathbf{b}$'yi çözmek iki ucuz üçgensel çözüme dönüşür:
önce $L\mathbf{y}=\mathbf{b}$, sonra $U\mathbf{x}=\mathbf{y}$. Hadi pivotsuz sürümü uygulayalım.

In [3]:
def lu_decompose(A):
    A = np.asarray(A, float).copy()
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = U[i, k] / U[k, k]      # çarpan
            L[i, k] = m                # onu L'de sakla
            U[i] = U[i] - m * U[k]     # pivotun altını ele
    return L, U

A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])
L, U = lu_decompose(A)
print("L =\n", L, "\n\nU =\n", U)
check("A == L U", L @ U, A)

L =
 [[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]] 

U =
 [[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
[PASS] A == L U                         max|Δ| = 0.00e+00


True

In [4]:
def forward_substitution(L, b):
    L, b = np.asarray(L, float), np.asarray(b, float)
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y

def lu_solve(A, b):
    L, U = lu_decompose(A)
    y = forward_substitution(L, b)     # L y = b
    return back_substitution(U, y)     # U x = y

b = np.array([4.0, 10.0, 24.0])
x = lu_solve(A, b)
check("lu_solve vs np.linalg.solve", x, np.linalg.solve(A, b))
print("çözüm x =", x)

[PASS] lu_solve vs np.linalg.solve      max|Δ| = 7.77e-16
solution x = [1. 1. 1.]


## 3. Pivotlama neden önemli: kararsızlık gerçektir

Pivotsuz algoritma, pivot $U_{kk}$'ye böler. Bir pivot çok küçükse, çarpanlar patlar ve
kayan-nokta hatası cevabı boğar. Çözüm — **kısmi pivotlama** — her adımda mevcut en büyük girdiyi
pivot konumuna takas eder. Üretim kodunun, bir permütasyon $P$ ile $PA = LU$ döndürmesinin nedeni
budur.

Aşağıdaki örnekte felaket derecede küçük bir pivot var; pivotsuz çözümün kayışını, pivotlu kütüphane
sonucu doğru kalırken izleyin.

In [5]:
eps = 1e-15
A_bad = np.array([[eps, 1.0],
                  [1.0, 1.0]])
b = np.array([1.0, 2.0])

x_naive = lu_solve(A_bad, b)             # pivotlama yok
x_ref   = np.linalg.solve(A_bad, b)      # pivotlu, güvenilir

print("naif (pivot yok) :", x_naive)
print("referans (pivot) :", x_ref)
print("gerçek cevap ≈ [1, 1]'dir; naif pivotsuz sonuç yanlıştır.")

# scipy'nin PA = LU'su, bunu düzelten permütasyonu gösterir
P, L, U = scipy_lu(A_bad)
print("\npermütasyon P (satırlar takas edildi):\n", P)

naive (no pivot) : [0.9992 1.    ]
reference (pivot): [1. 1.]
the true answer is ≈ [1, 1]; the naive pivot-free result is wrong.

permutation P (rows were swapped):
 [[0. 1.]
 [1. 0.]]


## 4. Koşullanma — cevap ne kadar duyarlı?

Mükemmel pivotlamayla bile, bazı matrisler özünde kırılgandır: $\mathbf{b}$'deki küçük bir değişim
$\mathbf{x}$'te büyük bir değişim üretir. **Koşul sayısı** $\kappa(A)$ bunu niceler. Büyük bir
$\kappa$, "kötü koşullanmış" demektir — sistem hataları büyütür. Bu tam niceliğin, Hafta 6'da
en büyük tekil değerin en küçüğe oranı olarak geri döndüğünü göreceğiz.

In [6]:
well = np.array([[2.0, 0.0], [0.0, 2.0]])
ill  = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-6]])

for name, M in [("iyi koşullanmış", well), ("kötü koşullanmış", ill)]:
    print(f"{name:<18} κ(A) = {np.linalg.cond(M):.3e}")
print("\nBüyük κ ⇒ çözüm, verideki gürültüye aşırı duyarlıdır.")

well-conditioned   κ(A) = 1.000e+00
ill-conditioned    κ(A) = 4.000e+06

Large κ ⇒ the solution is hypersensitive to noise in the data.


## 5. Bunun makine öğrenmesinde karşımıza çıktığı yer

- **Normal denklemler** $X^\top X \beta = X^\top y$ (sonraki ders, lineer regresyon) tam olarak
  bu üçgensel makineyle çözülür.
- **Koşullanma, eğitim acısını açıklar.** Kötü koşullanmış $X^\top X$ — korelasyonlu öznitelikler
  ya da kötü ölçeklemenin neden olduğu — regresyonu kararsız ve gradyan inişini yavaş yapar; öznitelik
  standartlaştırmasını ve ridge düzenlileştirmesini motive eder.
- **LU/Cholesky çarpanlamaları**, lineer sistemleri tekrar tekrar çözmesi gereken Gauss süreçlerinin,
  Kalman filtrelerinin ve ikinci-dereceden optimize edicilerin temelini oluşturur.

In [7]:
# zamanlama sezgisi: bir kez çarpanla, birçok sağ-taraf vektörünü ucuza çöz
A = np.random.default_rng(0).normal(size=(200, 200))
A = A @ A.T + 200 * np.eye(200)          # onu güzelce koşullandır (SPD)
L, U = lu_decompose(A)

B = np.random.default_rng(1).normal(size=(200, 5))   # 5 farklı hedef
X = np.column_stack([back_substitution(U, forward_substitution(L, b)) for b in B.T])
check("çoklu-RHS çözümü", A @ X, B, atol=1e-6)
print("5 sağ-taraf vektöründe yeniden kullanılan tek bir çarpanlama — LU'nun getirisi.")

[PASS] multi-RHS solve                  max|Δ| = 5.11e-15
One factorization reused across 5 right-hand sides — the LU payoff.


## Alıştırmalar

1. **İşlemleri sayın.** Bir $n\times n$ matrisi elemek kabaca kaç çarpma–toplama gerektirir? (İpucu: $n^3$ gibi ölçeklenir.)
2. **Elle pivot.** `lu_decompose` içinde kısmi pivotlamayı uygulayın (bir permütasyon izleyin) ve §3'teki kararsız örneği yeniden çözün.
3. **U'dan determinant.** Elemeden sonra, $\det A = \pm\prod_i U_{ii}$. Bunu rastgele bir matriste doğrulayın (işaret, satır takaslarının sayısından gelir).

In [8]:
# === Çözümler (önce kendiniz deneyin!) ===

# 1. ~ (2/3) n^3 çarpma-toplama → "Gauss eleme O(n^3)'tür".

# 3. pivotların çarpımı olarak determinant (pivotlama yok → işaret burada +1)
A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
L, U = lu_decompose(A)
det_from_U = np.prod(np.diag(U))
check("pivotlardan det", det_from_U, np.linalg.det(A), atol=1e-6)
print("U'nun köşegeninin çarpımı =", round(det_from_U, 6))

[PASS] det from pivots                  max|Δ| = 8.88e-16
product of U's diagonal = 4.0


## Özet ve sırada ne var

Eleme, herhangi bir sistemi üçgensel yapar; çarpanlar $L$'yi oluşturur ve sonuç $U$'dur, bu da
$A = LU$'yu verir. Pivotlama onu sayısal olarak güvende tutar ve koşul sayısı, cevaba ne kadar
güveneceğimizi söyler.

**Sırada — `06_en_kucuk_kareler_normal_denklemler.ipynb`:** gerçek verilerde bilinmeyenlerden daha fazla
denklem vardır, bu yüzden *kesin* bir çözüm yoktur. Hedefi sütun uzayına izdüşüreceğiz — Ders 2'nin
izdüşümünü Ders 4'ün sütun uzayına bağlayarak — ve ortaya **lineer regresyon** çıkar.

---
*Makine Öğrenmesi için Lineer Cebir · Hafta 3 · Ders 5*